# Level 2 — Lesson 4: Our First 3D Segmentation Model with MONAI

**AI for Radiotherapy**

**Teaching time:** ~30–45 minutes  
**Core question:** *Can we now combine everything we learned into a meaningful medical-imaging AI application?*

This lesson deliberately introduces only a few new ideas.

We will train a small 3D model to segment one structure from a CT volume.

```text
medical-image files
        ↓
MONAI transforms
        ↓
training patches
        ↓
3D U-Net
        ↓
voxel-wise prediction
        ↓
Dice loss
        ↓
training
        ↓
sliding-window inference
        ↓
3D segmentation
```

The goal is **not** state-of-the-art performance.

The goal is to build a complete segmentation workflow and recognize how much of it we already understand from Lessons 1–3.

## What is new?

Only four ideas are new:

1. segmentation produces a prediction **for every voxel**;
2. we use a segmentation-specific loss and metric;
3. training can happen on patches while inference happens on a whole volume;
4. **sliding-window inference** lets us process images that are too large to fit through the network at once.

Everything else should look familiar:

- problem definition and data quality → Lesson 1;
- PyTorch learning loop → Lesson 2;
- MONAI transforms, dictionaries, sampling, and `DataLoader` → Lesson 3.

> **This lesson is the payoff: the pieces now become a complete medical-imaging AI application.**

# 0. Setup

In [4]:
!pip -q install "monai==1.6.1" nibabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 25.0 MB/s eta 0:00:00


In [5]:
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

import torch
from torch import nn

import monai
from monai.apps import DecathlonDataset
from monai.data import CacheDataset, Dataset, DataLoader
from monai.inferers import sliding_window_inference
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    RandCropByPosNegLabeld,
    RandFlipd,
    EnsureTyped,
)
from monai.utils import set_determinism

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_determinism(seed=SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("MONAI:", monai.__version__)
print("PyTorch:", torch.__version__)
print("Device:", device)

MONAI: 1.6.1
PyTorch: 2.11.0+cpu
Device: cpu


# 1. Choose the data route

The **preferred route** uses the real Medical Segmentation Decathlon **Task09 Spleen** dataset.

It is a public 3D CT segmentation dataset and works directly with MONAI's `DecathlonDataset`.

If downloading the real data is inconvenient because of teaching time or network access, you can switch to the **synthetic fallback**.

The rest of the notebook works **identically** for both routes.

Both produce the same case structure:

```python
train_files = [
    {"image": "...", "label": "..."},
    ...
]

val_files = [
    {"image": "...", "label": "..."},
    ...
]
```

That is an important design principle for machine-learning projects:

> **The data source can change while the rest of the pipeline keeps the same interface.**

Keeping file structures and data interfaces simple and generalizable makes experiments easier to reproduce, modify, and extend.

In [6]:
#@title Data source { display-mode: "form" }

USE_REAL_DATA = True #@param {type:"boolean"}

# Small subsets keep the tutorial suitable for free Colab.
N_REAL_TRAIN_CASES = 6 #@param {type:"integer"}
N_REAL_VAL_CASES = 1 #@param {type:"integer"}

DATA_MODE = "real" if USE_REAL_DATA else "synthetic"

print("Requested data mode:", DATA_MODE)

Requested data mode: real


## Option A — real CT data

If `USE_REAL_DATA=True`, the next cell:

1. downloads MSD Task09 Spleen if it is not already present in the current Colab runtime;
2. prepares the training and validation split;
3. selects a small subset for this short tutorial.

The original dataset contains substantially more data than we need for a 30–45 minute teaching exercise.

> The first download may take a few minutes — a good opportunity for a short coffee or snack break.

If the real-data setup fails, the notebook automatically switches to the synthetic fallback.

In [7]:
#@title Download real CT data (~1.6 GB) — run this cell { display-mode: "form" }

REAL_DATA_AVAILABLE = False

if USE_REAL_DATA:
    try:
        real_root = Path("/content/msd")
        real_root.mkdir(parents=True, exist_ok=True)

        # Reuse the dataset if it is already present in the current Colab runtime.
        spleen_dir = real_root / "Task09_Spleen"
        need_download = not spleen_dir.exists()

        print("Data directory:", real_root)
        print(
            "Task09_Spleen already present in this runtime."
            if not need_download
            else "Task09_Spleen not found; downloading..."
        )

        real_train_dataset = DecathlonDataset(
            root_dir=real_root,
            task="Task09_Spleen",
            section="training",
            transform=None,
            download=need_download,
            seed=SEED,
            val_frac=0.2,
            cache_rate=0.0,
        )

        real_val_dataset = DecathlonDataset(
            root_dir=real_root,
            task="Task09_Spleen",
            section="validation",
            transform=None,
            download=False,
            seed=SEED,
            val_frac=0.2,
            cache_rate=0.0,
        )

        train_files = list(real_train_dataset.data)[:N_REAL_TRAIN_CASES]
        val_files = list(real_val_dataset.data)[:N_REAL_VAL_CASES]

        if len(train_files) == 0 or len(val_files) == 0:
            raise RuntimeError("No training or validation cases were found.")

        REAL_DATA_AVAILABLE = True
        DATA_MODE = "real"

        print("Using REAL MSD Spleen data")
        print("Training cases:", len(train_files))
        print("Validation cases:", len(val_files))

    except Exception as error:
        print("\nReal-data setup was not successful.")
        print("Reason:", error)
        print("\nSwitching to the synthetic fallback.")
        REAL_DATA_AVAILABLE = False
        DATA_MODE = "synthetic"

Data directory: /content/msd
Task09_Spleen not found; downloading once...


Task09_Spleen.tar: 1.50GB [01:29, 18.0MB/s]                            


2026-10-04 20:03:11,930 - INFO - Verified 'Task09_Spleen.tar', md5: 410d4a301da4e5b2f6f86ec3ddba524e.
2026-10-04 20:03:11,932 - INFO - Downloaded: /content/msd/Task09_Spleen.tar
2026-10-04 20:03:11,934 - INFO - Writing into directory: /content/msd.
Using REAL MSD Spleen data
Training cases: 6
Validation cases: 1


## Option B — synthetic fallback

The next cell creates a small CT-like dataset **only if the real-data route is not being used**.

The helper code is hidden because it is not part of the learning objective. It simply creates NIfTI files containing:

- CT-like intensities;
- one binary structure;
- realistic affine metadata;
- small case-to-case variation.

You can safely run the cell without studying its implementation.

In [ ]:
#@title Synthetic fallback generator — run this cell { display-mode: "form" }

if DATA_MODE == "synthetic":

    DATA_DIR = Path("/content/monai_lesson4_synthetic")
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    def create_synthetic_case(case_id, spacing, seed):
        rng = np.random.default_rng(seed)

        shape = (64, 64, 32)
        x, y, z = np.indices(shape)

        shift_x = int(rng.integers(-4, 5))
        shift_y = int(rng.integers(-4, 5))
        shift_z = int(rng.integers(-2, 3))

        ct = np.full(shape, -1000.0, dtype=np.float32)

        body = (
            ((x - (32 + shift_x)) / 27) ** 2
            + ((y - (32 + shift_y)) / 24) ** 2
            + ((z - (16 + shift_z)) / 14) ** 2
        ) < 1

        ct[body] = 25.0 + rng.normal(0, 30, size=body.sum())

        bone = (
            ((x - (32 + shift_x)) / 6) ** 2
            + ((y - (32 + shift_y)) / 5) ** 2
            + ((z - (16 + shift_z)) / 11) ** 2
        ) < 1

        ct[bone] = 650.0 + rng.normal(0, 60, size=bone.sum())

        organ = (
            ((x - (43 + shift_x)) / 8) ** 2
            + ((y - (32 + shift_y)) / 10) ** 2
            + ((z - (17 + shift_z)) / 7) ** 2
        ) < 1

        ct[organ] = 70.0 + rng.normal(0, 12, size=organ.sum())
        label = organ.astype(np.uint8)

        sx, sy, sz = spacing

        affine = np.array([
            [-sx, 0.0, 0.0, (shape[0] - 1) * sx],
            [0.0, -sy, 0.0, (shape[1] - 1) * sy],
            [0.0, 0.0, sz, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ])

        image_path = DATA_DIR / f"case_{case_id:02d}_ct.nii.gz"
        label_path = DATA_DIR / f"case_{case_id:02d}_label.nii.gz"

        nib.save(nib.Nifti1Image(ct, affine), image_path)
        nib.save(nib.Nifti1Image(label, affine), label_path)

        return {
            "image": str(image_path),
            "label": str(label_path),
        }

    spacings = [
        (1.2, 1.2, 2.5),
        (1.0, 1.0, 2.0),
        (1.3, 1.3, 2.5),
        (1.1, 1.1, 2.0),
        (1.2, 1.2, 2.2),
        (1.0, 1.0, 2.4),
    ]

    files = [
        create_synthetic_case(i, spacing, SEED + i)
        for i, spacing in enumerate(spacings)
    ]

    train_files = files[:5]
    val_files = files[5:]

    print("Using SYNTHETIC teaching data")
    print("Training cases:", len(train_files))
    print("Validation cases:", len(val_files))

else:
    print("Real data are active; synthetic generation skipped.")

In [8]:
print("Final data mode:", DATA_MODE)
print("First training case:")
print(train_files[0])

Final data mode: real
First training case:
{'image': '/content/msd/Task09_Spleen/imagesTr/spleen_28.nii.gz', 'label': '/content/msd/Task09_Spleen/labelsTr/spleen_28.nii.gz'}


# 2. Reuse the MONAI pipeline from Lesson 3

There is intentionally very little new here.

For training we:

- load image and label;
- standardize orientation and spacing;
- scale the CT intensity range;
- sample foreground/background patches;
- apply a small random flip.

For validation we use only deterministic preprocessing.

The same transforms work for both the real and synthetic datasets because both use the same case dictionary structure.

In [9]:
# The MSD spleen tutorial commonly focuses on a soft-tissue CT window.
# This also works adequately for our synthetic fallback.
CT_MIN = -57
CT_MAX = 164

PATCH_SIZE = (64, 64, 32)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),

    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=CT_MIN,
        a_max=CT_MAX,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),

    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=PATCH_SIZE,
        pos=1,
        neg=1,
        num_samples=2,
        image_key="image",
        image_threshold=0,
    ),

    RandFlipd(
        keys=["image", "label"],
        prob=0.5,
        spatial_axis=0,
    ),

    EnsureTyped(keys=["image", "label"]),
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),

    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),

    ScaleIntensityRanged(
        keys=["image"],
        a_min=CT_MIN,
        a_max=CT_MAX,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),

    EnsureTyped(keys=["image", "label"]),
])

/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [ ]:
train_ds = CacheDataset(
    data=train_files,
    transform=train_transforms,
    cache_rate=1.0,
    num_workers=2,
)

val_ds = Dataset(
    data=val_files,
    transform=val_transforms,
)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
)

batch = next(iter(train_loader))

print("Training image batch:", batch["image"].shape)
print("Training label batch:", batch["label"].shape)

A batch such as:

```text
[2, 1, 64, 64, 32]
```

means:

```text
2 sampled patches
1 input channel
64 × 64 × 32 voxels
```

We deliberately train on patches because whole 3D CT volumes may be too large to process efficiently at once.

# 3. Segmentation changes the output

In Lesson 2, our classifier produced one prediction for an entire image:

```text
image
  ↓
model
  ↓
class scores
```

Segmentation produces one prediction **for every voxel**:

```text
3D CT
  ↓
model
  ↓
3D prediction map
```

For our binary task:

```text
input:
[batch, 1, X, Y, Z]

output:
[batch, 1, X, Y, Z]
```

The single output channel represents the **foreground structure** we want to segment.

Each output voxel contains a **logit**. Applying a sigmoid maps that logit to a model-assigned foreground probability.

# 4. Define a small 3D U-Net

A U-Net can be viewed at a high level as:

```text
input
  ↓
encoder
  ↓
compressed features
  ↓
decoder
  ↓
voxel-wise prediction
```

Skip connections help the decoder recover spatial detail from earlier layers.

For the core lesson we use MONAI's implementation so that the architecture itself does not become the main topic.

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(8, 16, 32),
    strides=(2, 2),
    num_res_units=1,
).to(device)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ),
)

Interpret the most important arguments:

```text
spatial_dims=3
    → use 3D convolutions

in_channels=1
    → CT only

out_channels=1
    → one binary foreground target

channels=(8, 16, 32)
    → learned feature maps at successive levels
```

The model is intentionally small.

# 5. Dice: loss and evaluation

For a segmentation task, we care about overlap between the predicted foreground \(P\) and reference foreground \(G\):

\[
Dice = \frac{2|P \cap G|}{|P| + |G|}
\]

```text
Dice = 1 → perfect overlap
Dice = 0 → no overlap
```

MONAI provides a differentiable Dice loss for training and a Dice metric for evaluating thresholded predictions.

In this notebook, we evaluate the Dice score of the **foreground structure**.

This reinforces an idea from Lesson 2:

> **The quantity we optimize and the way we evaluate a model are related, but they are not automatically the same thing.**

We return to this point later when we ask what segmentation quality actually means in radiotherapy.

In [ ]:
loss_function = DiceLoss(sigmoid=True)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

# predictions and labels are single-channel label maps:
# 0 = background, 1 = foreground.
# num_classes=2 makes that interpretation explicit.
dice_metric = DiceMetric(
    include_background=False,
    reduction="mean",
    num_classes=2,
)

# 6. Sliding-window inference

Here is the practical problem:

```text
training
    model sees small patches

validation / application
    we want a prediction for the whole CT
```

Why not simply pass the full volume through the network?

A full 3D image may require much more GPU memory than a training patch. Since compute is limited, we often train with smaller regions.

At inference time, patch-based training is not a problem. We can divide the image into appropriately sized regions and apply the model to each region in turn.

**Sliding-window inference** automates this by moving a smaller window across the image:

```text
whole CT volume
      ↓
┌───────────┐
│ window 1  │ → prediction
└───────────┘

      move

        ┌───────────┐
        │ window 2  │ → prediction
        └───────────┘

      move ...
```

Predictions from overlapping windows are combined into one full-volume output.

This allows a patch-sized model to generate a prediction for a much larger image.

> **Patch-based training does not mean the final prediction has to be patch-sized.**

In [ ]:
ROI_SIZE = PATCH_SIZE
SW_BATCH_SIZE = 2

def predict_full_volume(model, images):
    return sliding_window_inference(
        inputs=images,
        roi_size=ROI_SIZE,
        sw_batch_size=SW_BATCH_SIZE,
        predictor=model,
        overlap=0.25,
    )

The `overlap` parameter means neighbouring windows share some voxels.

This helps reduce discontinuities at patch boundaries because each voxel can receive information from more than one window.

For this lesson, the important idea is the workflow rather than tuning the sliding-window parameters.

# 7. Train the model

The training loop should now look almost identical to Lesson 2.

The only important addition is that validation uses **whole-volume sliding-window inference**.

In [ ]:
# Real data are harder than the synthetic teaching problem.
# Keep the run deliberately short for a live tutorial.
N_EPOCHS = 6 if DATA_MODE == "synthetic" else 5

history = {
    "train_loss": [],
    "val_dice": [],
}

for epoch in range(1, N_EPOCHS + 1):

    # ------------------
    # Training on patches
    # ------------------
    model.train()
    running_loss = 0.0
    n_steps = 0

    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].float().to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        n_steps += 1

    epoch_loss = running_loss / n_steps
    history["train_loss"].append(epoch_loss)

    # ------------------
    # Whole-volume validation
    # ------------------
    model.eval()
    dice_metric.reset()

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device)
            labels = batch["label"].float().to(device)

            logits = predict_full_volume(
                model,
                images,
            )

            probabilities = torch.sigmoid(logits)
            predictions = (probabilities > 0.5).float()

            dice_metric(
                y_pred=predictions,
                y=labels,
            )

    val_dice = dice_metric.aggregate().item()
    history["val_dice"].append(val_dice)

    print(
        f"Epoch {epoch:02d}/{N_EPOCHS} | "
        f"train loss {epoch_loss:.3f} | "
        f"val Dice {val_dice:.3f}"
    )

Look again at the central training step:

```python
optimizer.zero_grad()
logits = model(images)
loss = loss_function(logits, labels)
loss.backward()
optimizer.step()
```

This is still PyTorch.

MONAI provided:

```text
medical-image transforms
patch sampling
U-Net implementation
Dice utilities
sliding-window inference
```

but the learning mechanism is the same one we studied in Lesson 2.

In [ ]:
epochs = np.arange(1, N_EPOCHS + 1)

plt.figure(figsize=(7, 4))
plt.plot(
    epochs,
    history["train_loss"],
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Dice loss")
plt.title("Training loss")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(
    epochs,
    history["val_dice"],
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Validation Dice")
plt.ylim(0, 1)
plt.title("Whole-volume validation Dice")
plt.show()

# 8. Inspect a whole-volume prediction

Numerical metrics are useful, but medical-image predictions should also be inspected visually.

We use sliding-window inference once more to produce the complete validation segmentation.

In [ ]:
model.eval()

val_case = val_ds[0]

image = val_case["image"].unsqueeze(0).to(device)
label = val_case["label"].unsqueeze(0).float().to(device)

with torch.no_grad():
    logits = predict_full_volume(
        model,
        image,
    )

    probability = torch.sigmoid(logits)
    prediction = (probability > 0.5).float()

image_np = image[0, 0].cpu().numpy()
label_np = label[0, 0].cpu().numpy()
pred_np = prediction[0, 0].cpu().numpy()

# Show the slice with the largest reference foreground area.
slice_areas = label_np.sum(axis=(0, 1))
z = int(np.argmax(slice_areas))

print("Displayed slice:", z)
print("Full prediction shape:", pred_np.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(
    image_np[:, :, z].T,
    cmap="gray",
    origin="lower",
)
axes[0].set_title("CT")

axes[1].imshow(
    image_np[:, :, z].T,
    cmap="gray",
    origin="lower",
)
axes[1].contour(
    label_np[:, :, z].T,
    levels=[0.5],
)
axes[1].set_title("Reference")

axes[2].imshow(
    image_np[:, :, z].T,
    cmap="gray",
    origin="lower",
)

if pred_np[:, :, z].max() > 0:
    axes[2].contour(
        pred_np[:, :, z].T,
        levels=[0.5],
    )

axes[2].set_title("Prediction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## What have we built?

Starting from CT and segmentation files, we have now:

```text
defined the problem
        ↓
chosen the representation
        ↓
preprocessed the images
        ↓
sampled training patches
        ↓
trained a 3D model
        ↓
performed whole-volume inference
        ↓
evaluated unseen data
        ↓
generated a 3D segmentation
```

That is a complete medical-imaging AI workflow.

The real-data route makes the task genuinely medical.  
The synthetic route keeps the same workflow available when compute or download constraints make the real dataset impractical.

# 9. From this notebook to one-click segmentation

A mature segmentation application may eventually present the user with:

```text
Load CT
   ↓
click "Apply"
   ↓
segmentation appears
```

Behind that simple interface are decisions about:

- training data;
- target definitions;
- preprocessing;
- model architecture;
- loss functions;
- sampling;
- optimization;
- validation;
- inference;
- post-processing;
- software integration.

> **The one-click experience is simple because substantial complexity was handled before deployment.**

# 10. Segmentation applied to radiotherapy

Now reflect on the model we designed in this notebook.

Are the data pipeline, augmentation, architecture, loss function, and evaluation metric appropriate for a radiotherapy application?

What would you change?

> **Before you move on, discuss with a partner or write down a a few things that come to mind.**

In radiotherapy, the segmentation is usually not the final product. It contributes to decisions that eventually affect the treatment delivered to the patient.

One simplified dependency chain might look like:

```text
Treatment quality
        ↑
Dose distribution
        ↑
Treatment plan
        ↑
Planning objectives / OAR weighting
        ↑
Volume definition
```

This is deliberately simplified — treatment quality also depends on prescription, beam geometry, robustness, delivery, anatomical change, and many other factors.

The important point is that **segmentation errors can propagate downstream**.

Geometric metrics such as Dice or Hausdorff distance are useful, but by themselves they do not capture the full clinical relevance of a segmentation error in radiotherapy.

Two contour errors with a similar geometric magnitude may have very different consequences.

For example:

- moving an OAR contour closer to a target may make it harder to satisfy the same planning objectives;
- moving it farther away may relax the apparent dose constraint;
- a small error near a steep dose gradient may matter more than a larger error in a low-gradient region.

So the clinically relevant question may eventually become:

> **How does segmentation uncertainty propagate into planning decisions, dose distributions, and treatment quality?**

For an example of propagating geometric uncertainty into downstream radiotherapy quantities, see:

A. Smolders et al. (2023), *Physics in Medicine & Biology* **68**, 245027.  
DOI: 10.1088/1361-6560/ad0282

The same task-aware thinking applies to **augmentation**.

Ask what variation you reasonably expect in unseen data, and whether the augmentation still represents a physically or clinically plausible example.

For example:

- In dose prediction, are arbitrary dose deformations physically meaningful?
- In beam-angle selection, does randomly cropping the CT remove information that the task requires?
- When predicting tiny structures or steep gradients, what information might aggressive downsampling destroy?
- We used a simple random flip in this tutorial. Would that be anatomically plausible for every radiotherapy task?

These decisions matter.

> **MONAI gives us many tools, but choosing a transform is still a modelling decision. It should be justified by the task and the data.**

# Optional exercise — replace MONAI's model with your own PyTorch model

So far we used MONAI's:

```python
model = UNet(...)
```

That is convenient, but MONAI models are still ordinary PyTorch `nn.Module` objects.

Anything that accepts:

```text
[batch, channel, X, Y, Z]
```

and returns:

```text
[batch, 1, X, Y, Z]
```

can be inserted into the same training and sliding-window inference code.

Let's make a deliberately tiny model ourselves.

In [ ]:
class TinySegNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=8,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),

            nn.Conv3d(
                in_channels=8,
                out_channels=8,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
        )

        self.output = nn.Conv3d(
            in_channels=8,
            out_channels=1,
            kernel_size=1,
        )

    def forward(self, x):
        x = self.features(x)
        return self.output(x)


tiny_model = TinySegNet().to(device)

print(tiny_model)
print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in tiny_model.parameters()
        if p.requires_grad
    ),
)

### Exercise

Replace the MONAI U-Net with:

```python
model = TinySegNet().to(device)
```

Create a new optimizer and rerun training.

Ask:

- Does the training loop still work?
- Does sliding-window inference still work?
- Is the model faster?
- Does it achieve the same Dice?
- Why might a U-Net have an advantage for segmentation?

Notice what did **not** change:

```text
MONAI transforms
Dataset
DataLoader
Dice loss
optimizer concept
training loop
sliding-window inference
validation
```

> **MONAI is still PyTorch. The lessons from Lesson 2 apply directly.**

## Optional next step — build a tiny encoder–decoder

If you want to explore architecture design, try constructing:

```text
Conv3d
↓
MaxPool3d
↓
Conv3d
↓
Upsample
↓
Conv3d
```

Then ask:

- Why does downsampling increase the spatial context available to deeper features?
- What spatial information may be lost?
- Why could skip connections help restore detail?

You have now reached the main ideas underlying a U-Net.

This is intentionally outside the core lesson.

# 11. Final reflection

Across four lessons we moved through the full chain:

```text
problem definition
→ data
→ representation
→ preprocessing
→ sampling
→ model
→ loss
→ optimization
→ inference
→ validation
→ interpretation
```

The architecture is only one part of that system.

When you watch a mature tool such as TotalSegmentator produce many structures with a single click, you now have a better idea of the work hidden behind that button:

```text
data
→ training
→ validation
→ inference
→ software integration
→ "Apply"
```

The interface may be simple. Building a model that deserves to sit behind it is not.

# Exit check

You should now be able to explain:

1. How segmentation differs from image classification.
2. Why a binary segmentation model can use one foreground output channel.
3. What the Dice coefficient measures.
4. Why training can use patches even when the desired output is a whole-volume segmentation.
5. What sliding-window inference does.
6. Why overlapping inference windows can be useful.
7. Why the PyTorch training loop from Lesson 2 still applies.
8. Why a MONAI U-Net can be replaced by another PyTorch `nn.Module`.
9. Why visualization and task-specific evaluation remain important even when a numerical metric is available.
10. What kinds of work are hidden behind a mature "one-click" segmentation application.

---

## References and further learning

- MONAI documentation: https://docs.monai.io/
- MONAI tutorials: https://github.com/Project-MONAI/tutorials
- Medical Segmentation Decathlon Task09 Spleen is a public 3D CT spleen-segmentation task used in MONAI examples.
- MONAI provides `sliding_window_inference` specifically for patch-wise inference over larger images.

For extended learning, the official MONAI 3D segmentation examples add:

- larger datasets;
- more complete validation;
- richer augmentation;
- post-processing;
- alternative losses and networks;
- more extensive inference workflows.

This notebook is educational material and is not intended for clinical use.